# 🎬 YouTube Video Notes Generator

## What We're Building

A tool that takes a **YouTube video URL**, fetches its transcript, and generates:
- **Chaptered notes** (organized by topic)
- **Key points** summary
- **Action items** (things to do or remember)

## How It Works

```
YouTube URL → Fetch transcript → Split into sections → LLM summarizes each section
                                                                    ↓
                                            Combine into structured notes
```

## Stack
- **youtube-transcript-api** — fetches video transcripts (no API key needed!)
- **LangChain** — prompt templates and chain orchestration
- **Ollama (qwen3.5:9b)** — local LLM for summarization

> **Note:** This only works for videos that have subtitles/closed captions.
> Most popular YouTube videos have auto-generated captions.

## Step 1 — Install & Import Dependencies

In [ ]:
# Install if needed (uncomment)
# !pip install youtube-transcript-api langchain langchain-ollama -q

from youtube_transcript_api import YouTubeTranscriptApi
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import re
import textwrap

print("All imports successful!")

## Step 2 — Fetch YouTube Transcript

The `youtube-transcript-api` library fetches subtitles directly from YouTube.
Each transcript entry has:
- `text` — the spoken words
- `start` — timestamp in seconds
- `duration` — how long this segment lasts

We'll extract the video ID from the URL and fetch whatever transcript is available.

In [ ]:
def extract_video_id(url: str) -> str:
    """Extract video ID from various YouTube URL formats."""
    patterns = [
        r'(?:v=|/v/|youtu\.be/)([\w-]{11})',  # Standard and short URLs
        r'(?:embed/)([\w-]{11})',               # Embed URLs
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Could not extract video ID from: {url}")


def get_transcript(video_url: str) -> list[dict]:
    """Fetch transcript for a YouTube video."""
    video_id = extract_video_id(video_url)
    print(f"Video ID: {video_id}")
    
    transcript = YouTubeTranscriptApi.get_transcript(
        video_id,
        languages=["en"],  # Prefer English
    )
    return transcript


def format_timestamp(seconds: float) -> str:
    """Convert seconds to MM:SS or HH:MM:SS format."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    if hours > 0:
        return f"{hours}:{minutes:02d}:{secs:02d}"
    return f"{minutes}:{secs:02d}"


# Test with a popular tech talk
# Replace this URL with any YouTube video that has captions
VIDEO_URL = "https://www.youtube.com/watch?v=aircAruvnKk"  # 3Blue1Brown - Neural Networks

transcript = get_transcript(VIDEO_URL)
print(f"\nFetched {len(transcript)} transcript segments")
print(f"\nFirst 5 segments:")
for seg in transcript[:5]:
    ts = format_timestamp(seg['start'])
    print(f"  [{ts}] {seg['text']}")

## Step 3 — Combine Transcript into Timed Sections

Raw transcript segments are tiny (a few words each). We group them into
**5-minute sections** to give the LLM enough context for meaningful summaries.

Each section retains its start timestamp so we can create "chapters."

In [ ]:
def group_transcript_into_sections(
    transcript: list[dict],
    section_duration: int = 300,  # 5 minutes in seconds
) -> list[dict]:
    """Group transcript segments into larger timed sections."""
    sections = []
    current_section = {"start": 0, "texts": []}
    section_start = 0
    
    for segment in transcript:
        # Start a new section if we've exceeded the duration
        if segment["start"] - section_start >= section_duration and current_section["texts"]:
            current_section["text"] = " ".join(current_section["texts"])
            sections.append(current_section)
            section_start = segment["start"]
            current_section = {"start": segment["start"], "texts": []}
        
        current_section["texts"].append(segment["text"])
    
    # Don't forget the last section
    if current_section["texts"]:
        current_section["text"] = " ".join(current_section["texts"])
        sections.append(current_section)
    
    return sections


sections = group_transcript_into_sections(transcript)
print(f"Grouped into {len(sections)} sections\n")

for i, section in enumerate(sections):
    ts = format_timestamp(section['start'])
    word_count = len(section['text'].split())
    print(f"Section {i+1} [{ts}]: {word_count} words")
    print(f"  Preview: {section['text'][:100]}...\n")

## Step 4 — Initialize the LLM

In [ ]:
llm = ChatOllama(
    model="qwen3.5:9b",
    temperature=0.3,
)

print("LLM initialized!")

## Step 5 — Summarize Each Section

We send each section to the LLM with a prompt asking for:
- A **chapter title** (2-5 words)
- **Key points** (bullet list)
- **Important details** worth noting

### Why summarize section-by-section?
- Gives us natural "chapter" boundaries
- Works even if the total transcript exceeds the LLM's context window
- Each section's summary is independent and more focused

In [ ]:
section_prompt = ChatPromptTemplate.from_template(
    """You are a note-taking assistant. Summarize this section of a video transcript.

Provide:
1. A short chapter title (2-5 words)
2. Key points as bullet points (3-5 bullets)
3. Any important terms, numbers, or quotes mentioned

Format your response exactly like this:
## [Chapter Title]
- Key point 1
- Key point 2
- Key point 3

**Notable:** [any important terms, numbers, or quotes]

Transcript section (starting at {timestamp}):
{text}
"""
)

# Build the chain: prompt → LLM → parse output as string
section_chain = section_prompt | llm | StrOutputParser()

# Summarize each section
section_summaries = []
for i, section in enumerate(sections):
    ts = format_timestamp(section["start"])
    print(f"Summarizing section {i+1}/{len(sections)} [{ts}]...")
    
    summary = section_chain.invoke({
        "timestamp": ts,
        "text": section["text"],
    })
    section_summaries.append({"timestamp": ts, "summary": summary})
    print(f"  ✓ Done")

print(f"\nAll {len(section_summaries)} sections summarized!")

## Step 6 — Generate Final Comprehensive Notes

Now we combine all section summaries and ask the LLM to produce
a final, organized set of notes with action items.

In [ ]:
# Combine section summaries
combined_summaries = "\n\n".join(
    f"[{s['timestamp']}]\n{s['summary']}" for s in section_summaries
)

# Final notes prompt
final_prompt = ChatPromptTemplate.from_template(
    """You are a study notes assistant. Given the section-by-section summaries of a video,
create comprehensive, well-organized notes.

Your output should include:

1. **OVERVIEW** — A 2-3 sentence summary of the entire video
2. **CHAPTERS** — Keep the chaptered summaries with timestamps
3. **KEY TAKEAWAYS** — The 5 most important points from the entire video
4. **ACTION ITEMS** — Things the viewer should do, try, or explore further
5. **GLOSSARY** — Define any technical terms mentioned

Section summaries:
{summaries}
"""
)

final_chain = final_prompt | llm | StrOutputParser()

print("Generating final notes...")
final_notes = final_chain.invoke({"summaries": combined_summaries})
print("Done!\n")
print("=" * 70)
print(final_notes)
print("=" * 70)

## Step 7 — Save Notes to File

In [ ]:
from pathlib import Path

output_path = Path("video_notes.md")
output_path.write_text(
    f"# Video Notes\n\n"
    f"**Source:** {VIDEO_URL}\n\n"
    f"---\n\n"
    f"{final_notes}",
    encoding="utf-8",
)
print(f"Notes saved to: {output_path.absolute()}")

## Step 8 — Interactive Mode (Try Your Own Videos!)

In [ ]:
def generate_notes(video_url: str) -> str:
    """Complete pipeline: URL → structured notes."""
    print(f"🎬 Processing: {video_url}")
    
    # 1. Fetch transcript
    transcript = get_transcript(video_url)
    print(f"  📝 Got {len(transcript)} transcript segments")
    
    # 2. Group into sections
    sections = group_transcript_into_sections(transcript)
    print(f"  📂 Grouped into {len(sections)} sections")
    
    # 3. Summarize each section
    summaries = []
    for i, section in enumerate(sections):
        ts = format_timestamp(section["start"])
        summary = section_chain.invoke({"timestamp": ts, "text": section["text"]})
        summaries.append({"timestamp": ts, "summary": summary})
        print(f"  ✓ Section {i+1}/{len(sections)}")
    
    # 4. Generate final notes
    combined = "\n\n".join(f"[{s['timestamp']}]\n{s['summary']}" for s in summaries)
    notes = final_chain.invoke({"summaries": combined})
    
    print(f"  ✅ Notes generated!")
    return notes


# Uncomment and try with your own video:
# my_notes = generate_notes("https://www.youtube.com/watch?v=YOUR_VIDEO_ID")
# print(my_notes)

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **youtube-transcript-api** | Fetches subtitles/captions from YouTube without an API key |
| **Section Grouping** | Combines tiny transcript segments into meaningful time-based sections |
| **ChatPromptTemplate** | Defines structured prompts with input variables |
| **StrOutputParser** | Extracts plain text from LLM response objects |
| **Chain (prompt \| llm \| parser)** | LangChain Expression Language (LCEL) for composing steps |
| **Map-Reduce Pattern** | Summarize each section individually, then combine all summaries |

## 🔧 Customization Ideas

- **Change section duration**: `section_duration=600` for 10-minute chapters
- **Different output format**: Modify the final prompt for flashcards, quiz questions, etc.
- **Multi-language**: Change `languages=["en"]` to support other languages
- **Batch processing**: Loop over a playlist of video URLs